# 📊 AyurVani — Data Processing Pipeline

Chunk classical Ayurveda texts and ingest them into the vector store.

In [ ]:
import os
import json
import glob
import boto3
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/ayurveda_corpus')
EMBED_DIR = Path('../data/embeddings')
EMBED_DIR.mkdir(exist_ok=True)

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
MODEL_ID = 'amazon.nova-embed-multimodal-v1:0'

print(f'Data dir: {DATA_DIR}')
print(f'Files found: {len(list(DATA_DIR.rglob("*.txt")))}')

In [ ]:
def chunk_text(text, chunk_size=512, overlap=64):
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def create_embedding(text):
    """Create embedding using Nova Multimodal Embeddings."""
    body = json.dumps({'inputText': text, 'dimensions': 1024, 'embeddingTypes': ['float']})
    response = bedrock.invoke_model(body=body, modelId=MODEL_ID, accept='application/json', contentType='application/json')
    result = json.loads(response['body'].read())
    return result['embedding']

# Test
test_emb = create_embedding('Ayurveda is the science of life')
print(f'Embedding dimension: {len(test_emb)}')

In [ ]:
# Process all text files
all_chunks = []

for txt_file in DATA_DIR.rglob('*.txt'):
    text = txt_file.read_text(encoding='utf-8')
    category = txt_file.parent.name
    chunks = chunk_text(text)
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'content': chunk,
            'source': f'{txt_file.name}:{i}',
            'metadata': {'category': category, 'file': txt_file.name}
        })

print(f'Total chunks: {len(all_chunks)}')

In [ ]:
# Generate and save embeddings
embeddings = []

for i, chunk in enumerate(all_chunks):
    emb = create_embedding(chunk['content'])
    embeddings.append(emb)
    chunk['embedding'] = emb
    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1}/{len(all_chunks)}')

np.save(EMBED_DIR / 'corpus_embeddings.npy', np.array(embeddings))

with open(EMBED_DIR / 'corpus_metadata.json', 'w') as f:
    json.dump([{k: v for k, v in c.items() if k != 'embedding'} for c in all_chunks], f, indent=2)

print('Done! Saved embeddings and metadata.')